# Chapter 19 — G-Retriever on ExplaGraphs (checkpoint producer)

**Requirements**: A100 or H100 GPU (Colab Pro / Pro+).

## Why ExplaGraphs

ExplaGraphs is a stance-detection dataset where the graph explicitly encodes the commonsense reasoning chain from belief to argument. The task is a clean binary classification (`support` / `counter`), so accuracy is unambiguous.

This is the dataset where the original G-Retriever paper shows the largest, most reproducible gap over text-only baselines (~85% vs lower baselines).

## Workflow

1. Cell 1-2: install + verify GPU.
2. Cell 3 with `SMOKE_TEST = True`: verifies the pipeline runs end-to-end.
3. Cell 3 with `SMOKE_TEST = False`: full 10-epoch training + per-epoch checkpoints.
4. Cell 4: comparison G-Retriever vs text-only baseline on the test set.
5. Cell 5 (optional): upload final checkpoint to HuggingFace Hub.

## 1. Install

In [1]:
!pip install -q 'git+https://github.com/pyg-team/pytorch_geometric.git@master'
!pip install -q -U torchao
!pip install -q transformers datasets sentencepiece accelerate peft

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 50.8 MB/s eta 0:00:00


## 2. Verify GPU

In [2]:
!nvidia-smi
import torch
assert torch.cuda.is_available(), 'No GPU detected.'
print(f'Device: {torch.cuda.get_device_name(0)}')
print(f'Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

Mon Oct  5 15:19:15 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA RTX PRO 6000 Blac...    Off |   00000000:05:00.0 Off |                    0 |
| N/A   33C    P0             46W /  600W |       0MiB /  97887MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

## 3. Training

First run with `SMOKE_TEST = True` (default). If `SMOKE TEST PASSED`, flip to `False` and re-run.

**Watch `val_acc` per epoch.** Baseline (untrained) should be around 0.5 (chance). If G-Retriever training works, expect val_acc to climb toward 0.7-0.85 over 10 epochs. If it stays around 0.5, the training is not learning and we need to change something.

In [3]:
"""
Chapter 19 — G-Retriever training on ExplaGraphs (checkpoint producer)

Why ExplaGraphs instead of WebQSP:
  - Binary task (stance = support/counter), so evaluation is exact accuracy
    with no keyword-matching heuristics needed.
  - The graph explicitly encodes the reasoning chain (belief → concepts →
    argument), so a GNN encoder has a genuine informational advantage
    over a text-only baseline reading the same graph as linear text.
  - The G-Retriever paper reports ~85% on this dataset with a strong gap
    over text-only baselines, so we have prior evidence the setup works.

TWO MODES controlled by SMOKE_TEST:
  True  → 1 epoch, tiny val batch, 1 sample: pipeline sanity check.
  False → 10 epochs, full evaluation, checkpoints per epoch.

ARTIFACTS under ./checkpoints/:
  - epoch_XX.pt      trainable weights per epoch
  - history.json     per-epoch train/val loss + accuracy + samples
  - timings.json     per-phase wall-clock times
"""

import math, os, time, json, re, csv
import io as io_
import urllib.request
import torch
import numpy as np
from contextlib import contextmanager
from typing import List, Optional

# ═══════════════════════════════════════════════════════════════════════════
#                          ↓ FLIP THIS FLAG ↓
SMOKE_TEST = False
# ═══════════════════════════════════════════════════════════════════════════

SEED = 0
torch.manual_seed(SEED); np.random.seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
assert device.type == "cuda", "This notebook requires a GPU."
print(f"Using device: {torch.cuda.get_device_name(0)}")
print(f"CUDA memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print(f"\nMODE: {'SMOKE TEST' if SMOKE_TEST else 'FULL TRAINING RUN'}")

# ── Timing ─────────────────────────────────────────────────────────────────
timings: dict = {}
CHECKPOINT_DIR = "./checkpoints"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
TIMINGS_PATH = os.path.join(CHECKPOINT_DIR, "timings.json")

@contextmanager
def timed(name: str):
    start = time.time()
    try:
        yield
    finally:
        timings[name] = time.time() - start
        with open(TIMINGS_PATH, "w") as f:
            json.dump(timings, f, indent=2)

def print_timings():
    print("\nTiming so far:")
    width = max(len(n) for n in timings) if timings else 0
    for name, s in timings.items():
        print(f"  {name:<{width}} : {s:>7.1f}s ({s/60:>5.2f} min)")

# ── Hyperparameters ────────────────────────────────────────────────────────
BATCH_SIZE    = 4
NUM_EPOCHS    = 1 if SMOKE_TEST else 10
LR            = 1e-5
WEIGHT_DECAY  = 0.05
WARMUP_EPOCHS = 0 if SMOKE_TEST else 1
MIN_LR        = 5e-6
GRAD_CLIP     = 0.1
N_VAL_MAX     = 8 if SMOKE_TEST else None
N_SAMPLES     = 1 if SMOKE_TEST else 3

def adjust_lr(pg, base_lr, epoch, total):
    if epoch < WARMUP_EPOCHS:
        lr = base_lr
    else:
        prog = (epoch - WARMUP_EPOCHS) / max(1, total - WARMUP_EPOCHS)
        lr = MIN_LR + (base_lr - MIN_LR) * 0.5 * (1 + math.cos(math.pi * prog))
    pg["lr"] = lr
    return lr


# ============================================================================
# ExplaGraphs dataset — custom loader
# ============================================================================
#
# We fetch two TSV files from the ExplaGraphs GitHub repo. Each row is
#     belief \t argument \t stance \t graph
# where `graph` is a bracketed string:
#     (c1; rel; c2)(c3; rel; c4)...
#
# Following the G-Retriever paper we use ExplaGraphs `train.tsv` as our
# combined pool, split it 80/20 for training/validation, and use the
# original `dev.tsv` as the held-out test set (the true test set is hidden).

from torch_geometric.data       import Data
from torch_geometric.llm.models import LLM, GRetriever, SentenceTransformer
from torch_geometric.nn         import GAT
from torch_geometric.loader     import DataLoader

EXPLA_BASE = "https://raw.githubusercontent.com/swarnaHub/ExplaGraphs/master/data"
DATA_DIR   = "./data/ExplaGraphs"
os.makedirs(DATA_DIR, exist_ok=True)

def download_expla_files():
    for name in ["train.tsv", "dev.tsv"]:
        local = os.path.join(DATA_DIR, name)
        if not os.path.exists(local):
            print(f"Downloading {name} …")
            urllib.request.urlretrieve(f"{EXPLA_BASE}/{name}", local)

def parse_graph(graph_str: str) -> List[tuple]:
    """Turn `(a; rel; b)(c; rel; d)…` into a list of (src, rel, dst) tuples."""
    triples = []
    for match in re.findall(r"\(([^)]+)\)", graph_str):
        parts = [p.strip() for p in match.split(";")]
        if len(parts) == 3:
            triples.append(tuple(parts))
    return triples

def read_tsv(path: str) -> list:
    """Load rows: belief, argument, stance, graph_string. Returns list of dicts."""
    rows = []
    with open(path, encoding="utf-8") as f:
        reader = csv.reader(f, delimiter="\t")
        for r in reader:
            if len(r) < 4: continue
            belief, argument, stance, graph_str = r[:4]
            if stance not in ("support", "counter"): continue
            triples = parse_graph(graph_str)
            if not triples: continue
            rows.append({
                "belief": belief.strip(),
                "argument": argument.strip(),
                "stance": stance.strip(),
                "triples": triples,
            })
    return rows


def build_dataset(rows: list, sbert: "SentenceTransformer") -> List[Data]:
    """Convert list of dict rows into a list of PyG Data objects.

    IMPORTANT: `question`, `label`, `desc` are stored as plain strings
    (not lists). WebQSPDataset uses this convention, and the PyG collate
    then concatenates them into flat lists per batch. Wrapping them in
    lists here would produce list-of-list batches and break the tokenizer.
    """
    dataset = []
    for row in rows:
        # Deduplicate nodes and relations within this sample
        nodes    = []
        node_ids = {}
        edges_src, edges_dst, edge_rels = [], [], []
        for src, rel, dst in row["triples"]:
            for txt in (src, dst):
                if txt not in node_ids:
                    node_ids[txt] = len(nodes)
                    nodes.append(txt)
            edges_src.append(node_ids[src])
            edges_dst.append(node_ids[dst])
            edge_rels.append(rel)

        # Embed nodes and edge relations with SentenceBERT
        x         = sbert.encode(nodes,     batch_size=64, output_device="cpu")
        edge_attr = sbert.encode(edge_rels, batch_size=64, output_device="cpu")

        # Undirected edge_index (both directions)
        src_t = torch.tensor(edges_src, dtype=torch.long)
        dst_t = torch.tensor(edges_dst, dtype=torch.long)
        edge_index = torch.stack(
            [torch.cat([src_t, dst_t]), torch.cat([dst_t, src_t])], dim=0)
        # Duplicate edge_attr for the reverse direction
        edge_attr = torch.cat([edge_attr, edge_attr], dim=0)

        # Textual context for the LLM: the graph as bracketed triples
        desc = " ".join(f"({s}; {r}; {d})" for s, r, d in row["triples"])

        # Prompt formulated so the answer is a single word: support or counter.
        # `/no_think` disables Qwen3's reasoning mode so the model answers
        # directly instead of emitting a <think>...</think> block.
        question = (
            f"/no_think Belief: {row['belief']}\n"
            f"Argument: {row['argument']}\n"
            f"Does the argument support or counter the belief? "
            f"Answer with a single word: support or counter."
        )

        dataset.append(Data(
            x=x, edge_index=edge_index, edge_attr=edge_attr,
            question=question, label=row["stance"], desc=desc,
        ))
    return dataset


# ============================================================================
# Load LLM and build the dataset
# ============================================================================

print("\n" + "=" * 60)
print("Loading Qwen3-0.6B and ExplaGraphs")
print("=" * 60)

with timed("load_qwen3"):
    llm = LLM(
        model_name="Qwen/Qwen3-0.6B",
        num_params=0.6,
        dtype=torch.bfloat16,
        sys_prompt=(
            "/no_think You are a helpful assistant. Given a belief, an "
            "argument, and a commonsense graph, decide whether the argument "
            "supports or counters the belief. Answer with a single word: "
            "support or counter."
        ),
    )

with timed("build_explagraphs"):
    download_expla_files()
    sbert = SentenceTransformer("sentence-transformers/all-roberta-large-v1").to(device)
    sbert.eval()

    train_rows = read_tsv(os.path.join(DATA_DIR, "train.tsv"))
    test_rows  = read_tsv(os.path.join(DATA_DIR, "dev.tsv"))

    # 80/20 split of train.tsv → train/val (deterministic thanks to seed)
    rng = np.random.default_rng(SEED)
    idx = rng.permutation(len(train_rows))
    split = int(0.8 * len(train_rows))
    train_slice = [train_rows[i] for i in idx[:split]]
    val_slice   = [train_rows[i] for i in idx[split:]]

    train_ds = build_dataset(train_slice, sbert)
    val_ds   = build_dataset(val_slice,   sbert)
    test_ds  = build_dataset(test_rows,   sbert)

print(f"train={len(train_ds)}, val={len(val_ds)}, test={len(test_ds)}")
print(f"sample: {train_ds[0]}")
print(f"labels in train: {set(d.label for d in train_ds)}")
print_timings()


# ============================================================================
# Assemble G-Retriever
# ============================================================================

print("\n" + "=" * 60)
print("Assembling G-Retriever (GAT + Qwen3-0.6B + LoRA)")
print("=" * 60)

with timed("assemble_gretriever"):
    gnn = GAT(
        in_channels=1024,
        hidden_channels=1024,
        num_layers=4,
        out_channels=1024,
        heads=4,
    )
    model = GRetriever(
        llm=llm, gnn=gnn,
        use_lora=True, mlp_out_tokens=1,
    ).to(device)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f"Trainable parameters: {trainable/1e6:.1f}M / {total/1e6:.1f}M "
      f"({100*trainable/total:.2f}%)")
print_timings()


# ============================================================================
# Training with per-epoch checkpoints + val accuracy
# ============================================================================

print("\n" + "=" * 60)
print(f"Training ({NUM_EPOCHS} epoch{'s' if NUM_EPOCHS != 1 else ''})")
print("=" * 60)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False)

optimizer = torch.optim.AdamW(
    [p for p in model.parameters() if p.requires_grad],
    lr=LR, weight_decay=WEIGHT_DECAY,
)


def compute_loss(batch):
    batch = batch.to(device)
    return model(
        question=batch.question, x=batch.x.float(),
        edge_index=batch.edge_index, batch=batch.batch,
        label=batch.label,
        edge_attr=(batch.edge_attr.float() if batch.edge_attr is not None else None),
        additional_text_context=batch.desc,
    )


@torch.no_grad()
def evaluate(loader, max_batches=None):
    """Returns (avg_loss, accuracy). Accuracy = fraction where the first
    token of the predicted answer matches the stance label."""
    model.eval()
    total_loss, n_seen, correct = 0.0, 0, 0
    for i, batch in enumerate(loader):
        if max_batches is not None and i >= max_batches: break
        loss = compute_loss(batch)
        total_loss += loss.item() * batch.num_graphs
        n_seen     += batch.num_graphs
        preds = model.inference(
            question=batch.question, x=batch.x.float().to(device),
            edge_index=batch.edge_index.to(device),
            batch=batch.batch.to(device),
            edge_attr=(batch.edge_attr.float().to(device)
                       if batch.edge_attr is not None else None),
            additional_text_context=batch.desc,
            max_out_tokens=24,
        )
        for pred, label in zip(preds, batch.label):
            # Strip Qwen3's <think>...</think> block if present
            answer = re.sub(r"<think>.*?</think>", "", pred, flags=re.DOTALL)
            words = answer.strip().lower().split()
            first_word = re.sub(r"[^a-z]", "", words[0]) if words else ""
            if first_word.startswith(label[:4].lower()):  # 'supp' or 'coun'
                correct += 1
    model.train()
    return total_loss / max(1, n_seen), correct / max(1, n_seen)


@torch.no_grad()
def sample_val(n=3):
    model.eval()
    out = []
    for i in range(min(n, len(val_ds))):
        sample = val_ds[i]
        pred = model.inference(
            question=[sample.question],
            x=sample.x.float().to(device),
            edge_index=sample.edge_index.to(device),
            batch=torch.zeros(sample.x.size(0), dtype=torch.long, device=device),
            edge_attr=(sample.edge_attr.float().to(device)
                       if sample.edge_attr is not None else None),
            additional_text_context=[sample.desc],
            max_out_tokens=24,
        )[0]
        # Strip <think>...</think> for cleaner display
        clean_pred = re.sub(r"<think>.*?</think>", "", pred, flags=re.DOTALL).strip()
        out.append({"belief_arg": sample.question.split("\n")[0][:80],
                    "expected": sample.label, "predicted": clean_pred[:60]})
    model.train()
    return out


history = []

with timed("training_total"):
    for epoch in range(NUM_EPOCHS):
        lr = adjust_lr(optimizer.param_groups[0], LR, epoch, NUM_EPOCHS)

        with timed(f"train_pass_epoch_{epoch+1:02d}"):
            model.train()
            train_loss, n_seen = 0.0, 0
            for batch in train_loader:
                optimizer.zero_grad()
                loss = compute_loss(batch)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(
                    [p for p in model.parameters() if p.requires_grad], GRAD_CLIP)
                optimizer.step()
                train_loss += loss.item() * batch.num_graphs
                n_seen     += batch.num_graphs
                if SMOKE_TEST and n_seen >= 4: break
            train_loss /= max(1, n_seen)

        with timed(f"val_eval_epoch_{epoch+1:02d}"):
            val_loss, val_acc = evaluate(val_loader, max_batches=N_VAL_MAX)

        with timed(f"samples_epoch_{epoch+1:02d}"):
            samples = sample_val(N_SAMPLES)

        history.append({
            "epoch": epoch + 1, "lr": lr,
            "train_loss": train_loss,
            "val_loss": val_loss, "val_acc": val_acc,
            "train_pass_s": timings[f"train_pass_epoch_{epoch+1:02d}"],
            "val_eval_s":   timings[f"val_eval_epoch_{epoch+1:02d}"],
            "samples_s":    timings[f"samples_epoch_{epoch+1:02d}"],
            "samples": samples,
        })

        print(f"\n--- Epoch {epoch+1}/{NUM_EPOCHS} ---")
        print(f"  lr={lr:.2e}  train_loss={train_loss:.4f}  "
              f"val_loss={val_loss:.4f}  val_acc={val_acc:.3f}")
        print(f"  train_pass={history[-1]['train_pass_s']:.1f}s  "
              f"val={history[-1]['val_eval_s']:.1f}s  "
              f"samples={history[-1]['samples_s']:.1f}s")
        for i, s in enumerate(samples):
            print(f"  [{i+1}] {s['belief_arg']}")
            print(f"      Expected:  {s['expected']}")
            print(f"      Predicted: {s['predicted']}")

        suffix = "smoketest" if SMOKE_TEST else f"epoch_{epoch+1:02d}"
        ckpt_path = os.path.join(CHECKPOINT_DIR, f"{suffix}.pt")
        state = model.state_dict()
        param_grad = {k: v.requires_grad for k, v in model.named_parameters()}
        for k in list(state.keys()):
            if k in param_grad and not param_grad[k]:
                del state[k]
        torch.save({
            "epoch": epoch + 1, "state_dict": state,
            "train_loss": train_loss, "val_loss": val_loss, "val_acc": val_acc,
        }, ckpt_path)
        print(f"  Saved: {ckpt_path} ({os.path.getsize(ckpt_path)/1e6:.1f} MB)")

        with open(os.path.join(CHECKPOINT_DIR, "history.json"), "w") as f:
            json.dump(history, f, indent=2, default=str)

print_timings()

if not SMOKE_TEST and len(history) >= 3:
    initial_acc = history[0]["val_acc"]
    latest_acc  = history[-1]["val_acc"]
    delta = latest_acc - initial_acc
    print(f"\nVal accuracy went from {initial_acc:.3f} to {latest_acc:.3f} "
          f"({delta:+.3f}).")

if SMOKE_TEST:
    print("\n" + "=" * 60)
    print("SMOKE TEST PASSED")
    print("=" * 60)
    print("Set SMOKE_TEST = False at the top of this cell and re-run.")
else:
    print("\nDone. Artifacts under ./checkpoints/")


Using device: NVIDIA RTX PRO 6000 Blackwell Server Edition
CUDA memory: 102.0 GB

MODE: FULL TRAINING RUN

Loading Qwen3-0.6B and ExplaGraphs
Setting up 'Qwen/Qwen3-0.6B' with configuration: {'revision': 'main', 'max_memory': {0: '94GiB'}, 'low_cpu_mem_usage': True, 'device_map': 'auto', 'dtype': torch.bfloat16}


config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.73k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.42GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

train=1894, val=474, test=398
sample: Data(x=[5, 1024], edge_index=[2, 8], edge_attr=[8, 1024], question='/no_think Belief: Refusal to use gender-neutral language excludes many people.
Argument: There are many recognized genders, and the only equitable way to treat them all is to use gender-neutral language.
Does the argument support or counter the belief? Answer with a single word: support or counter.', label='support', desc='(refusal gender-neutral; capable of; excludes many people) (excludes many people; capable of; bad) (gender-neutral language; has context; equitable) (equitable; not has context; bad)')
labels in train: {'support', 'counter'}

Timing so far:
  load_qwen3        :     9.3s ( 0.16 min)
  build_explagraphs :    39.8s ( 0.66 min)

Assembling G-Retriever (GAT + Qwen3-0.6B + LoRA)
Trainable parameters: 10.6M / 606.7M (1.75%)

Timing so far:
  load_qwen3          :     9.3s ( 0.16 min)
  build_explagraphs   :    39.8s ( 0.66 min)
  assemble_gretriever :     0.4s ( 0.01 m

In [4]:
# =============================================================================
# Comparison: G-Retriever vs Text-only baseline on the ExplaGraphs test set
# =============================================================================
#
# Same LLM (Qwen3-0.6B + LoRA), same input graph, same context. The only
# difference: G-Retriever also fuses a GNN-encoded soft prompt from the
# graph structure, while the text-only baseline sees the graph as
# linearized triples in plain text.

import re, json
import torch

model.eval()

def clean_prediction(pred: str) -> str:
    """Strip Qwen3's <think>...</think> block and normalize."""
    answer = re.sub(r"<think>.*?</think>", "", pred, flags=re.DOTALL)
    words = answer.strip().lower().split()
    if not words:
        return ""
    return re.sub(r"[^a-z]", "", words[0])

def match_stance(pred: str, label: str) -> bool:
    """`support` matches `supp*`, `counter` matches `coun*`."""
    return pred.startswith(label[:4].lower())


results = {"gretriever": [], "text_only": []}
transcripts = []

with torch.no_grad():
    for i, sample in enumerate(test_ds):
        # --- 1. G-Retriever: GNN soft prompt + text context ------------
        gr_raw = model.inference(
            question=[sample.question],
            x=sample.x.float().to(device),
            edge_index=sample.edge_index.to(device),
            batch=torch.zeros(sample.x.size(0), dtype=torch.long, device=device),
            edge_attr=(sample.edge_attr.float().to(device)
                       if sample.edge_attr is not None else None),
            additional_text_context=[sample.desc],
            max_out_tokens=24,
        )[0]

        # --- 2. Text-only baseline: same LLM, same context, NO GNN -----
        text_prompt = (
            f"{sample.question}\n\n"
            f"Graph context:\n{sample.desc}\n\n"
            f"Answer:"
        )
        text_raw = model.llm.inference([text_prompt], max_tokens=24)[0]

        gr_clean   = clean_prediction(gr_raw)
        text_clean = clean_prediction(text_raw)

        results["gretriever"].append(match_stance(gr_clean, sample.label))
        results["text_only"].append(match_stance(text_clean, sample.label))

        transcripts.append({
            "belief":    sample.question.split("\n")[0][:80],
            "expected":  sample.label,
            "gretriever": gr_clean,
            "text_only":  text_clean,
        })

        if (i + 1) % 50 == 0:
            print(f"  processed {i+1}/{len(test_ds)}")

# --- Summary --------------------------------------------------------------
print("\n" + "=" * 70)
print(f"Comparison on {len(test_ds)} test samples")
print("=" * 70)
for method, hits in results.items():
    correct = sum(hits)
    pct = 100 * correct / len(hits)
    print(f"  {method:<12}: {correct:>3}/{len(hits)}   ({pct:5.1f}%)")

both_correct = sum(1 for i in range(len(test_ds))
                   if results["gretriever"][i] and results["text_only"][i])
gr_only      = sum(1 for i in range(len(test_ds))
                   if results["gretriever"][i] and not results["text_only"][i])
text_only    = sum(1 for i in range(len(test_ds))
                   if not results["gretriever"][i] and results["text_only"][i])
both_wrong   = sum(1 for i in range(len(test_ds))
                   if not results["gretriever"][i] and not results["text_only"][i])

print(f"\nBoth correct:        {both_correct:>3}")
print(f"G-Retriever only:    {gr_only:>3}")
print(f"Text-only only:      {text_only:>3}")
print(f"Both wrong:          {both_wrong:>3}")

with open("./checkpoints/comparison.json", "w") as f:
    json.dump({"results": results, "transcripts": transcripts}, f,
              indent=2, default=str)
print("\nSaved comparison to ./checkpoints/comparison.json")

print("\n" + "=" * 70)
print("First 10 test samples side-by-side")
print("=" * 70)
for i, t in enumerate(transcripts[:10]):
    gr_ok   = "✓" if results["gretriever"][i] else "✗"
    text_ok = "✓" if results["text_only"][i]  else "✗"
    print(f"\n[{i+1}] {t['belief']}")
    print(f"    Expected:    {t['expected']}")
    print(f"    G-Retriever: {gr_ok} {t['gretriever']}")
    print(f"    Text-only:   {text_ok} {t['text_only']}")

  processed 50/398
  processed 100/398
  processed 150/398
  processed 200/398
  processed 250/398
  processed 300/398
  processed 350/398

Comparison on 398 test samples
  gretriever  : 349/398   ( 87.7%)
  text_only   : 260/398   ( 65.3%)

Both correct:        249
G-Retriever only:    100
Text-only only:       11
Both wrong:           38

Saved comparison to ./checkpoints/comparison.json

First 10 test samples side-by-side

[1] /no_think Belief: marriage is pase.
    Expected:    support
    G-Retriever: ✗ counter
    Text-only:   ✓ support

[2] /no_think Belief: marriage is the best for a family unit.
    Expected:    support
    G-Retriever: ✓ support
    Text-only:   ✓ support

[3] /no_think Belief: Entrapment shouldn't be legalized since it puts people into fa
    Expected:    support
    G-Retriever: ✓ support
    Text-only:   ✓ support

[4] /no_think Belief: Marriage is the backbone of society.
    Expected:    support
    G-Retriever: ✓ support
    Text-only:   ✓ support

[5] 